# BEST-Rec v2.2: Fixing the Fundamental MAE Problem

## Why MAE is 1.2 (and why that's worse than predicting the mean)

On Amazon Beauty, the average rating is ~4.2 and most ratings are 4 or 5. Predicting the global mean for everyone would give MAE ≈ 0.7-0.8. Our model at 1.2 is doing **worse than a constant predictor**. Here's why:

### Root cause analysis:

**1. GroupKFold holds out entire users.** ~20% of users have ZERO signal — no reviews, no bias, no SVD. Their user embedding is all zeros. The model produces garbage predictions for them.

**2. No user-side collaborative factors.** v2 only computes item SVD factors. Users get no collaborative signal at all — their entire representation depends on review text. No text = no signal.

**3. The model doesn't know it's blind.** When user features are zeros, the neural network still produces a prediction from that noise. It should instead fall back to a simple "item popularity" prediction.

**4. SmoothL1 loss under-penalises large errors.** On a 1-5 scale, SmoothL1 treats a 2-star error linearly. We need MSE to aggressively correct large mistakes.

### What v2.2 fixes:

| Fix | Expected MAE impact |
|---|---|
| **User SVD factors** — give every training user a collaborative vector | -0.15 to -0.25 |
| **Cold-user gating** — detect zero-signal users, auto-switch to item-only path | -0.10 to -0.20 |
| **Bounded output** — `1 + 4*sigmoid(x)` forces output into [1,5] during training | -0.05 to -0.10 |
| **MSE loss** — aggressively penalise large errors | -0.05 to -0.10 |
| **Direct rating prior** — pass raw item avg_rating as a strong feature | -0.05 to -0.10 |

**Target: MAE ≤ 0.65, which would beat the global-mean baseline and match collaborative filtering.**

## 0. Dependencies

In [ ]:
import subprocess, sys
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
pip_install("torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu128")
pip_install("transformers", "scikit-learn", "scipy", "numpy", "tqdm", "pandas", "ipywidgets")
print("Done.")

## 1. Imports

In [ ]:
import os, json, pickle, copy, time, warnings
from collections import defaultdict
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix
from transformers import AutoTokenizer, AutoModel

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
else:
    vram_gb = 0
    print("CPU mode")

## 2. Configuration

In [ ]:
DATASET = "beauty"

DATASET_FILES = {
    "beauty":      ("All_Beauty.jsonl",          "meta_All_Beauty.jsonl"),
    "books":       ("Books.jsonl",               "meta_Books.jsonl"),
    "fashion":     ("Amazon_Fashion.jsonl",      "meta_Amazon_Fashion.jsonl"),
    "instruments": ("Musical_Instruments.jsonl", "meta_Musical_Instruments.jsonl"),
}

DATA_DIR  = "./data"
CACHE_DIR = f"./cache/{DATASET}"
os.makedirs(CACHE_DIR, exist_ok=True)

INTER_FILE, META_FILE = DATASET_FILES[DATASET]
INTER_PATH = os.path.join(DATA_DIR, DATASET, INTER_FILE)
META_PATH  = os.path.join(DATA_DIR, DATASET, META_FILE)

# Hardware
NUM_CPU_WORKERS = min(8, max(0, os.cpu_count() - 2))
USE_AMP    = device.type == "cuda"
PIN_MEMORY = device.type == "cuda"
BATCH_SIZE = 4096 if vram_gb >= 12 else (2048 if vram_gb >= 8 else 1024)

# Model
PRETRAINED_MODEL   = "distilbert-base-uncased"
TEXT_DIM           = 768
HIDDEN_DIM         = 256     # smaller — bias does the heavy lifting now
NUM_HEADS          = 4
NUM_ENCODER_LAYERS = 1       # 1 layer is enough with bias terms
MAX_USER_REVIEWS   = 5
SVD_COMPONENTS     = 128     # smaller, denser signal
USER_SVD_COMPONENTS = 128    # NEW: user-side SVD
NUM_CLASSES        = 5

# Training
LR           = 5e-4
EPOCHS       = 50
PATIENCE     = 10
LAMBDA_CLS   = 0.5          # reduced — regression is more important
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
WARMUP_EPOCHS = 3

# Eval
NUM_FOLDS           = 5
NEG_SAMPLES         = 99
TOP_K               = 10
COLD_USER_THRESHOLD = 3
COLD_ITEM_THRESHOLD = 5
RANKING_EVAL_USERS  = 2000

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Dataset: {DATASET}  |  BS: {BATCH_SIZE}  |  H: {HIDDEN_DIM}")
print(f"SVD: item={SVD_COMPONENTS}, user={USER_SVD_COMPONENTS}")

## 3. Cache + Load Data

In [ ]:
def cached(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"  Computing: {name}...")
    result = fn()
    with open(path, "wb") as f:
        pickle.dump(result, f)
    return result

def cached_tensor(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        return torch.load(path, weights_only=True)
    print(f"  Computing: {name}...")
    result = fn()
    torch.save(result, path)
    return result

# Load raw data (from v2)
raw_path = os.path.join(CACHE_DIR, "raw_data.pkl")
if not os.path.exists(raw_path):
    print("ERROR: Run v2 notebook first to create cached data!")
    raise FileNotFoundError(raw_path)

with open(raw_path, "rb") as f:
    data = pickle.load(f)

interactions  = data["interactions"]
item_metadata = data["item_metadata"]
num_users     = data["num_users"]
num_items     = data["num_items"]

# Precomputed features (no leakage — metadata only)
item_title_embeds = cached_tensor("item_title_embeds.pt", lambda: None)

def _compute_numeric():
    avg_r = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)])
    r_num = torch.tensor([item_metadata[i]["rating_num"] for i in range(num_items)])
    price = torch.tensor([item_metadata[i]["price"]      for i in range(num_items)])
    def zscore(x): return (x - x.mean()) / (x.std() + 1e-9)
    return torch.stack([zscore(avg_r), zscore(r_num), zscore(price)], dim=-1)

item_numeric = cached_tensor("item_numeric.pt", _compute_numeric)

# NEW: raw item average rating (un-normalized) as a strong prior
item_raw_avg = torch.tensor(
    [item_metadata[i]["avg_rating"] for i in range(num_items)],
    dtype=torch.float32
)
# Replace 0.0 (missing) with global mean
global_avg = item_raw_avg[item_raw_avg > 0].mean().item()
item_raw_avg[item_raw_avg == 0] = global_avg

print(f"Users: {num_users:,}  Items: {num_items:,}  Interactions: {len(interactions):,}")
print(f"Global avg rating: {global_avg:.3f}")

## 4. Diagnostic: What Should MAE Be?

Before training any model, let's establish baselines to understand the target.

In [ ]:
ratings_arr = np.array([i["rating"] for i in interactions])

# Baseline 1: predict global mean for everyone
global_mean = ratings_arr.mean()
mae_global = np.mean(np.abs(ratings_arr - global_mean))
print(f"Global mean ({global_mean:.3f}) → MAE = {mae_global:.4f}")

# Baseline 2: predict item mean
item_means = defaultdict(list)
for inter in interactions:
    item_means[inter["item_id"]].append(inter["rating"])
item_mean_dict = {k: np.mean(v) for k, v in item_means.items()}

preds_item_mean = np.array([item_mean_dict.get(i["item_id"], global_mean) for i in interactions])
mae_item_mean = np.mean(np.abs(ratings_arr - preds_item_mean))
print(f"Item mean        → MAE = {mae_item_mean:.4f}")

# Baseline 3: predict user mean
user_means = defaultdict(list)
for inter in interactions:
    user_means[inter["user_id"]].append(inter["rating"])
user_mean_dict = {k: np.mean(v) for k, v in user_means.items()}

preds_user_mean = np.array([user_mean_dict.get(i["user_id"], global_mean) for i in interactions])
mae_user_mean = np.mean(np.abs(ratings_arr - preds_user_mean))
print(f"User mean        → MAE = {mae_user_mean:.4f}")

# Baseline 4: user_mean + item_mean - global_mean
preds_combined = np.array([
    user_mean_dict.get(i["user_id"], global_mean) +
    item_mean_dict.get(i["item_id"], global_mean) -
    global_mean
    for i in interactions
])
preds_combined = np.clip(preds_combined, 1, 5)
mae_combined = np.mean(np.abs(ratings_arr - preds_combined))
print(f"User+Item mean   → MAE = {mae_combined:.4f}")

print(f"\n→ Our model MUST beat {mae_global:.3f} (global mean).")
print(f"→ Target: beat {mae_combined:.3f} (user+item mean) with neural features.")
print(f"\nRating distribution:")
for r in [1,2,3,4,5]:
    c = int((ratings_arr == r).sum())
    print(f"  {r}: {c:>10,} ({100*c/len(ratings_arr):.1f}%)")

## 5. Per-Fold Feature Engineering

### New: User SVD factors

v2 only computed item SVD factors. v2.2 also computes **user SVD factors** from the user×item matrix, giving every training user a collaborative signal even without reviews.

In [ ]:
class TextEncoder:
    def __init__(self, model_name, device, max_length=64):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        self.device = device
        self.max_length = max_length
        self.dim = self.model.config.hidden_size

    @torch.no_grad()
    def encode_batch(self, texts, batch_size=256):
        all_e = []
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else "empty" for t in texts[i:i+batch_size]]
            inp = self.tokenizer(batch, padding=True, truncation=True,
                                  max_length=self.max_length, return_tensors="pt").to(self.device)
            with autocast(enabled=USE_AMP):
                out = self.model(**inp)
            all_e.append(out.last_hidden_state[:, 0, :].float().cpu())
        return torch.cat(all_e)

    def encode_user_reviews_for_fold(self, train_inters, num_users, max_reviews=5):
        user_reviews = defaultdict(list)
        for inter in train_inters:
            if inter["review"].strip():
                user_reviews[inter["user_id"]].append(inter["review"])
        triples = []
        for uid in range(num_users):
            for j, rev in enumerate(user_reviews.get(uid, [])[:max_reviews]):
                triples.append((uid, j, rev))
        embeds = torch.zeros(num_users, max_reviews, self.dim)
        masks  = torch.zeros(num_users, max_reviews, dtype=torch.bool)
        if triples:
            enc = self.encode_batch([t[2] for t in triples])
            for idx, (uid, j, _) in enumerate(triples):
                embeds[uid, j] = enc[idx]
                masks[uid, j]  = True
        return embeds, masks

text_encoder = TextEncoder(PRETRAINED_MODEL, device)


def compute_fold_features(train_inters, fold_tag, force=False):
    """Compute ALL per-fold features in one function: item SVD, user SVD, user text embeds."""

    # Item SVD
    def _item_svd():
        rows, cols, vals = [], [], []
        for inter in train_inters:
            rows.append(inter["item_id"]); cols.append(inter["user_id"]); vals.append(inter["rating"])
        mat = csr_matrix((vals, (rows, cols)), shape=(num_items, num_users))
        k = min(SVD_COMPONENTS, min(num_items, num_users) - 1, len(set(rows)) - 1)
        if k < 1: return torch.zeros(num_items, SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        print(f"    Item SVD: k={k}, var={svd.explained_variance_ratio_.sum():.4f}")
        if k < SVD_COMPONENTS: res = np.hstack([res, np.zeros((num_items, SVD_COMPONENTS - k))])
        return torch.tensor(res, dtype=torch.float32)

    item_svd = cached_tensor(f"svd_fold_{fold_tag}.pt", _item_svd, force)

    # NEW: User SVD
    def _user_svd():
        rows, cols, vals = [], [], []
        for inter in train_inters:
            rows.append(inter["user_id"]); cols.append(inter["item_id"]); vals.append(inter["rating"])
        mat = csr_matrix((vals, (rows, cols)), shape=(num_users, num_items))
        k = min(USER_SVD_COMPONENTS, min(num_users, num_items) - 1, len(set(rows)) - 1)
        if k < 1: return torch.zeros(num_users, USER_SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        print(f"    User SVD: k={k}, var={svd.explained_variance_ratio_.sum():.4f}")
        if k < USER_SVD_COMPONENTS: res = np.hstack([res, np.zeros((num_users, USER_SVD_COMPONENTS - k))])
        return torch.tensor(res, dtype=torch.float32)

    user_svd = cached_tensor(f"user_svd_fold_{fold_tag}.pt", _user_svd, force)

    # User text embeddings
    def _user_text():
        e, m = text_encoder.encode_user_reviews_for_fold(train_inters, num_users, MAX_USER_REVIEWS)
        return {"embeds": e, "masks": m}
    ut = cached(f"user_embeds_fold_{fold_tag}.pkl", _user_text, force)

    # Rating statistics for bias init
    global_sum, global_n = 0.0, 0
    user_sums, user_counts = defaultdict(float), defaultdict(int)
    item_sums, item_counts = defaultdict(float), defaultdict(int)
    for inter in train_inters:
        r = inter["rating"]
        global_sum += r; global_n += 1
        user_sums[inter["user_id"]] += r; user_counts[inter["user_id"]] += 1
        item_sums[inter["item_id"]] += r; item_counts[inter["item_id"]] += 1
    g_mean = global_sum / global_n

    u_bias = torch.zeros(num_users)
    i_bias = torch.zeros(num_items)
    for uid in range(num_users):
        if user_counts[uid] > 0:
            u_bias[uid] = (user_sums[uid] / user_counts[uid]) - g_mean
    for iid in range(num_items):
        if item_counts[iid] > 0:
            i_bias[iid] = (item_sums[iid] / item_counts[iid]) - g_mean

    # Track which users have signal
    user_has_signal = torch.zeros(num_users, dtype=torch.bool)
    for uid in range(num_users):
        has_reviews = ut["masks"][uid].any().item()
        has_interactions = user_counts[uid] > 0
        user_has_signal[uid] = has_reviews or has_interactions

    return {
        "item_svd": item_svd,
        "user_svd": user_svd,
        "user_text_embeds": ut["embeds"],
        "user_text_masks": ut["masks"],
        "global_mean": g_mean,
        "user_bias": u_bias,
        "item_bias": i_bias,
        "user_has_signal": user_has_signal,
    }

print("Feature engineering ready (now includes user SVD).")

## 6. Dataset Class (with user SVD + signal flag)

In [ ]:
class BESTRecDatasetV22(Dataset):
    def __init__(self, interactions, fold_feats, item_title_embeds, item_numeric, item_raw_avg):
        self.user_ids   = torch.tensor([i["user_id"] for i in interactions], dtype=torch.long)
        self.item_ids   = torch.tensor([i["item_id"] for i in interactions], dtype=torch.long)
        self.ratings    = torch.tensor([i["rating"]  for i in interactions], dtype=torch.float32)
        self.cls_labels = torch.clamp(self.ratings.long() - 1, min=0)

        self.user_text_embeds = fold_feats["user_text_embeds"]
        self.user_text_masks  = fold_feats["user_text_masks"]
        self.user_svd         = fold_feats["user_svd"]
        self.user_has_signal  = fold_feats["user_has_signal"]
        self.item_svd         = fold_feats["item_svd"]
        self.item_title_embeds = item_title_embeds
        self.item_numeric      = item_numeric
        self.item_raw_avg      = item_raw_avg

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        uid = self.user_ids[idx]; iid = self.item_ids[idx]
        return (
            self.user_text_embeds[uid], self.user_text_masks[uid],
            self.user_svd[uid], self.user_has_signal[uid],
            self.item_title_embeds[iid], self.item_numeric[iid],
            self.item_svd[iid], self.item_raw_avg[iid],
            self.ratings[idx], self.cls_labels[idx], uid, iid,
        )

## 7. BEST-Rec v2.2 Model

### Key design: Two-path architecture with gated fusion

```
Path 1 (always available): bias_pred = global_mean + user_bias + item_bias
Path 2 (when user has signal): neural_residual from cross-attention

Final = gate * (bias + neural) + (1 - gate) * bias_only_with_item_features
```

The gate is learned and depends on whether the user has any signal. For cold users (no reviews, no training interactions), the model gracefully falls back to item-only prediction.

In [ ]:
class BESTRecV22(nn.Module):
    def __init__(self, num_users, num_items, global_mean, user_bias_init, item_bias_init):
        super().__init__()
        H = HIDDEN_DIM

        # ── Bias terms ──
        self.global_mean = nn.Parameter(torch.tensor(float(global_mean)), requires_grad=False)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.user_bias.weight.data = user_bias_init.unsqueeze(1)
        self.item_bias.weight.data = item_bias_init.unsqueeze(1)

        # ── User projections ──
        self.user_review_proj = nn.Linear(TEXT_DIM, H)
        self.user_svd_proj    = nn.Linear(USER_SVD_COMPONENTS, H)

        # ── Item projections ──
        self.item_title_proj   = nn.Linear(TEXT_DIM, H)
        self.item_numeric_proj = nn.Linear(3, H)
        self.item_svd_proj     = nn.Linear(SVD_COMPONENTS, H)
        self.item_token_type   = nn.Embedding(3, H)

        # ── User attention pooling ──
        self.user_pool_query = nn.Parameter(torch.randn(1, 1, H) * 0.02)
        self.user_pool_attn  = nn.MultiheadAttention(embed_dim=H, num_heads=NUM_HEADS,
                                                      batch_first=True, dropout=0.1)

        # ── Per-tower encoders (lightweight: 1 layer) ──
        u_layer = nn.TransformerEncoderLayer(d_model=H, nhead=NUM_HEADS,
                    dim_feedforward=H*4, dropout=0.1, batch_first=True, activation="gelu")
        self.user_encoder = nn.TransformerEncoder(u_layer, num_layers=NUM_ENCODER_LAYERS)

        i_layer = nn.TransformerEncoderLayer(d_model=H, nhead=NUM_HEADS,
                    dim_feedforward=H*4, dropout=0.1, batch_first=True, activation="gelu")
        self.item_encoder = nn.TransformerEncoder(i_layer, num_layers=NUM_ENCODER_LAYERS)

        # ── Cross-attention (bidirectional) ──
        self.cross_u2i = nn.MultiheadAttention(embed_dim=H, num_heads=NUM_HEADS,
                                                batch_first=True, dropout=0.1)
        self.norm_u2i  = nn.LayerNorm(H)
        self.cross_i2u = nn.MultiheadAttention(embed_dim=H, num_heads=NUM_HEADS,
                                                batch_first=True, dropout=0.1)
        self.norm_i2u  = nn.LayerNorm(H)

        # ── Residual fusion ──
        self.fusion_proj  = nn.Linear(H * 4, H)
        self.fusion_norm  = nn.LayerNorm(H)
        self.fusion_mlp   = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1))
        self.fusion_norm2 = nn.LayerNorm(H)

        # ── Neural residual head (bounded output) ──
        self.residual_head = nn.Sequential(
            nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(H // 2, 1))

        # ── Item-only path (for cold users) ──
        self.item_only_head = nn.Sequential(
            nn.Linear(H + 1, H // 2), nn.GELU(),  # +1 for raw avg rating
            nn.Linear(H // 2, 1))

        # ── Confidence gate: learned scalar per-sample ──
        self.gate_net = nn.Sequential(
            nn.Linear(H * 4 + 1, 64), nn.GELU(),  # +1 for has_signal flag
            nn.Linear(64, 1), nn.Sigmoid())

        # ── Classifier ──
        self.classifier = nn.Linear(H, NUM_CLASSES)

    def forward(self, user_reviews, user_mask, user_svd, user_has_signal,
                item_title, item_numeric, item_svd, item_raw_avg,
                user_ids=None, item_ids=None):
        B = user_reviews.size(0)
        H = HIDDEN_DIM

        # ── User tokens: reviews + SVD ──
        u_rev_tokens = self.user_review_proj(user_reviews)  # (B, max_rev, H)
        u_svd_token  = self.user_svd_proj(F.normalize(user_svd, p=2, dim=-1)).unsqueeze(1)  # (B, 1, H)

        # Append SVD as an extra token (always available for training users)
        u_tokens = torch.cat([u_rev_tokens, u_svd_token], dim=1)  # (B, max_rev+1, H)

        # Mask: reviews use user_mask, SVD token is always unmasked
        svd_unmask = torch.ones(B, 1, dtype=torch.bool, device=user_mask.device)
        u_full_mask = torch.cat([user_mask, svd_unmask], dim=1)  # (B, max_rev+1)
        u_pad_mask  = ~u_full_mask

        # Handle fully-padded users
        all_pad = u_pad_mask.all(dim=1)
        if all_pad.any():
            u_pad_mask[all_pad, -1] = False  # unmask SVD token at minimum

        u_enc = self.user_encoder(u_tokens, src_key_padding_mask=u_pad_mask)

        # Attention pooling
        pool_q = self.user_pool_query.expand(B, -1, -1)
        u_pool, _ = self.user_pool_attn(query=pool_q, key=u_enc, value=u_enc,
                                         key_padding_mask=u_pad_mask)
        u_pool = u_pool.squeeze(1)

        # ── Item tokens ──
        t_title = self.item_title_proj(item_title).unsqueeze(1)
        t_num   = self.item_numeric_proj(item_numeric).unsqueeze(1)
        t_svd   = self.item_svd_proj(F.normalize(item_svd, p=2, dim=-1)).unsqueeze(1)

        i_tokens = torch.cat([t_title, t_num, t_svd], dim=1)
        type_ids = torch.tensor([0,1,2], device=i_tokens.device).unsqueeze(0).expand(B,-1)
        i_tokens = i_tokens + self.item_token_type(type_ids)
        i_enc = self.item_encoder(i_tokens)
        i_pool = i_enc.mean(dim=1)

        # ── Cross-attention ──
        u2i, _ = self.cross_u2i(query=u_enc, key=i_enc, value=i_enc)
        u2i = self.norm_u2i(u2i + u_enc)
        u_mask_f = u_full_mask.unsqueeze(-1).float()
        u2i_pool = (u2i * u_mask_f).sum(1) / (u_mask_f.sum(1) + 1e-9)

        i2u, _ = self.cross_i2u(query=i_enc, key=u_enc, value=u_enc, key_padding_mask=u_pad_mask)
        i2u = self.norm_i2u(i2u + i_enc)
        i2u_pool = i2u.mean(dim=1)

        # ── Fusion ──
        fused_cat = torch.cat([u_pool, u2i_pool, i_pool, i2u_pool], dim=-1)
        z = self.fusion_norm(self.fusion_proj(fused_cat))
        z = self.fusion_norm2(z + self.fusion_mlp(z))

        # ── Neural residual (bounded to [-2, +2] via tanh) ──
        neural_raw = self.residual_head(z).squeeze(-1)
        neural_residual = 2.0 * torch.tanh(neural_raw)  # bounded [-2, 2]

        # ── Item-only prediction path ──
        item_only_input = torch.cat([i_pool, item_raw_avg.unsqueeze(-1)], dim=-1)
        item_only_residual = self.item_only_head(item_only_input).squeeze(-1)

        # ── Bias prediction ──
        if user_ids is not None and item_ids is not None:
            u_b = self.user_bias(user_ids).squeeze(-1)
            i_b = self.item_bias(item_ids).squeeze(-1)
            bias_pred = self.global_mean + u_b + i_b
        else:
            bias_pred = self.global_mean + torch.zeros(B, device=user_reviews.device)

        # ── Gated combination ──
        has_sig = user_has_signal.float().unsqueeze(-1)  # (B, 1)
        gate_input = torch.cat([fused_cat, has_sig], dim=-1)
        gate = self.gate_net(gate_input).squeeze(-1)  # (B,) in [0, 1]

        # Full prediction: gate between neural path and item-only path
        full_pred   = bias_pred + neural_residual
        simple_pred = bias_pred + item_only_residual
        rating_pred = gate * full_pred + (1 - gate) * simple_pred

        # Clamp during training too (bounded output)
        rating_pred = torch.clamp(rating_pred, 1.0, 5.0)

        cls_logits = self.classifier(z)
        return rating_pred, cls_logits

print("BESTRecV22 defined (user SVD + gated cold-user fallback + bounded output).")

## 8. Training & Evaluation

In [ ]:
amp_scaler = GradScaler(enabled=USE_AMP)


def get_scheduler(optimizer, steps_per_epoch):
    warmup = WARMUP_EPOCHS * steps_per_epoch
    total  = EPOCHS * steps_per_epoch
    def lr_lambda(step):
        if step < warmup: return step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, optimizer, scheduler, dataloader):
    global amp_scaler
    model.train()
    crit_reg = nn.MSELoss()  # MSE instead of SmoothL1 — penalise large errors
    crit_cls = nn.CrossEntropyLoss(label_smoothing=0.05)
    total_loss, n = 0.0, 0

    for batch in tqdm(dataloader, desc="  Train", leave=False):
        (u_rev, u_mask, u_svd, u_sig,
         i_tit, i_num, i_svd, i_avg,
         rat, cls, uids, iids) = batch

        u_rev  = u_rev.to(device, non_blocking=True)
        u_mask = u_mask.to(device, non_blocking=True)
        u_svd  = u_svd.to(device, non_blocking=True)
        u_sig  = u_sig.to(device, non_blocking=True)
        i_tit  = i_tit.to(device, non_blocking=True)
        i_num  = i_num.to(device, non_blocking=True)
        i_svd  = i_svd.to(device, non_blocking=True)
        i_avg  = i_avg.to(device, non_blocking=True)
        rat    = rat.to(device, non_blocking=True)
        cls    = cls.to(device, non_blocking=True)
        uids   = uids.to(device, non_blocking=True)
        iids   = iids.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            pred_r, pred_cls = model(u_rev, u_mask, u_svd, u_sig,
                                      i_tit, i_num, i_svd, i_avg,
                                      user_ids=uids, item_ids=iids)
            loss = crit_reg(pred_r, rat) + LAMBDA_CLS * crit_cls(pred_cls, cls)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        scheduler.step()
        total_loss += loss.item() * rat.size(0)
        n += rat.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate_rating(model, dataloader):
    model.eval()
    all_p, all_t = [], []
    for batch in tqdm(dataloader, desc="  Eval", leave=False):
        (u_rev, u_mask, u_svd, u_sig,
         i_tit, i_num, i_svd, i_avg,
         rat, cls, uids, iids) = batch
        with autocast(enabled=USE_AMP):
            pred, _ = model(
                u_rev.to(device,non_blocking=True), u_mask.to(device,non_blocking=True),
                u_svd.to(device,non_blocking=True), u_sig.to(device,non_blocking=True),
                i_tit.to(device,non_blocking=True), i_num.to(device,non_blocking=True),
                i_svd.to(device,non_blocking=True), i_avg.to(device,non_blocking=True),
                user_ids=uids.to(device,non_blocking=True), item_ids=iids.to(device,non_blocking=True))
        all_p.append(pred.float().cpu().numpy())
        all_t.append(rat.numpy())
    p, t = np.concatenate(all_p), np.concatenate(all_t)
    return mean_absolute_error(t, p), np.sqrt(mean_squared_error(t, p))


@torch.no_grad()
def evaluate_ranking(model, test_inters, fold_feats, item_title_embeds, item_numeric, item_raw_avg):
    model.eval()
    ut_items = defaultdict(set)
    for inter in test_inters: ut_items[inter["user_id"]].add(inter["item_id"])
    ndcg_list, hr_list = [], []
    all_items = set(range(num_items))
    rng = np.random.RandomState(SEED)
    users = [u for u in ut_items if ut_items[u]][:RANKING_EVAL_USERS]
    for uid in tqdm(users, desc="  Ranking", leave=False):
        for pos in ut_items[uid]:
            neg_pool = list(all_items - ut_items[uid])
            if len(neg_pool) < NEG_SAMPLES: continue
            negs = rng.choice(neg_pool, NEG_SAMPLES, replace=False)
            cands = [pos] + list(negs)
            n = len(cands)
            with autocast(enabled=USE_AMP):
                scores, _ = model(
                    fold_feats["user_text_embeds"][uid].unsqueeze(0).expand(n,-1,-1).to(device),
                    fold_feats["user_text_masks"][uid].unsqueeze(0).expand(n,-1).to(device),
                    fold_feats["user_svd"][uid].unsqueeze(0).expand(n,-1).to(device),
                    fold_feats["user_has_signal"][uid].unsqueeze(0).expand(n).to(device),
                    item_title_embeds[cands].to(device),
                    item_numeric[cands].to(device),
                    fold_feats["item_svd"][cands].to(device),
                    item_raw_avg[cands].to(device))
            scores = scores.float().cpu().numpy()
            ranked = np.argsort(-scores)
            pr = int(np.where(ranked == 0)[0][0])
            hr_list.append(1.0 if pr < TOP_K else 0.0)
            ndcg_list.append(1.0/np.log2(pr+2) if pr < TOP_K else 0.0)
    return {f"NDCG@{TOP_K}": np.mean(ndcg_list) if ndcg_list else 0.0,
            f"HR@{TOP_K}": np.mean(hr_list) if hr_list else 0.0}

print("Training functions ready (MSE loss + label smoothing).")

## 9. Run Fold

In [ ]:
def run_fold_v22(fold_tag, train_inters, test_inters):
    global amp_scaler
    amp_scaler = GradScaler(enabled=USE_AMP)

    print(f"\n{'='*60}")
    print(f"  [{fold_tag}]: {len(train_inters):,} train / {len(test_inters):,} test")
    print(f"{'='*60}")

    feats = compute_fold_features(train_inters, fold_tag)

    train_ds = BESTRecDatasetV22(train_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    test_ds  = BESTRecDatasetV22(test_inters,  feats, item_title_embeds, item_numeric, item_raw_avg)

    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_CPU_WORKERS,
                 pin_memory=PIN_MEMORY,
                 prefetch_factor=4 if NUM_CPU_WORKERS > 0 else None,
                 persistent_workers=True if NUM_CPU_WORKERS > 0 else False)
    train_dl = DataLoader(train_ds, shuffle=True,  **dl_kw)
    test_dl  = DataLoader(test_ds,  shuffle=False, **dl_kw)

    model = BESTRecV22(num_users, num_items,
                        feats["global_mean"], feats["user_bias"], feats["item_bias"]).to(device)

    raw_model = model
    if hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("  Compiled")
        except: pass

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, len(train_dl))

    best_mae, best_state, pat = float("inf"), None, 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        loss = train_one_epoch(model, optimizer, scheduler, train_dl)
        mae, rmse = evaluate_rating(model, test_dl)
        dt = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]
        mk = ""
        if mae < best_mae:
            best_mae = mae; best_state = copy.deepcopy(raw_model.state_dict()); pat = 0; mk = " *"
        else: pat += 1
        print(f"  Ep {epoch+1:2d} | loss={loss:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f} | lr={lr_now:.1e} | {dt:.0f}s{mk}")
        if pat >= PATIENCE:
            print(f"  Early stop at {epoch+1}"); break

    raw_model.load_state_dict(best_state)
    print(f"  Best MAE: {best_mae:.4f}")

    final_mae, final_rmse = evaluate_rating(model, test_dl)
    rank = evaluate_ranking(model, test_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    results = {"mae": final_mae, "rmse": final_rmse, **rank}
    for k, v in results.items(): print(f"     {k}: {v:.4f}")

    del model, raw_model, optimizer, scheduler, train_dl, test_dl
    torch.cuda.empty_cache()
    return results

## 10. Run All Folds

In [ ]:
print("=" * 60)
print("  BEST-Rec v2.2 — GroupKFold by user")
print("=" * 60)

user_ids_arr = np.array([i["user_id"] for i in interactions])
indices = np.arange(len(interactions))
gkf = GroupKFold(n_splits=NUM_FOLDS)

v22_results = []
for fold_idx, (tr_idx, te_idx) in enumerate(gkf.split(indices, groups=user_ids_arr)):
    tr = [interactions[i] for i in tr_idx]
    te = [interactions[i] for i in te_idx]
    v22_results.append(run_fold_v22(f"v22_warm_{fold_idx}", tr, te))

print("\n" + "=" * 60)
print("v2.2 RESULTS")
print("=" * 60)
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in v22_results]
    print(f"  {m:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}   {[round(v,4) for v in vals]}")

## 11. Cold-Start + Comparison

In [ ]:
# Cold-start
def split_cold(inters, key, threshold):
    counts = defaultdict(int)
    for i in inters: counts[i[key]] += 1
    cold = {k for k, c in counts.items() if c <= threshold}
    tr = [i for i in inters if i[key] not in cold]
    te = [i for i in inters if i[key] in cold]
    return tr, te

cu_tr, cu_te = split_cold(interactions, "user_id", COLD_USER_THRESHOLD)
print(f"Cold users: {len(cu_te):,} test interactions")
cold_user = run_fold_v22("v22_cold_user", cu_tr, cu_te) if len(cu_te) >= 10 else None

ci_tr, ci_te = split_cold(interactions, "item_id", COLD_ITEM_THRESHOLD)
print(f"Cold items: {len(ci_te):,} test interactions")
cold_item = run_fold_v22("v22_cold_item", ci_tr, ci_te) if len(ci_te) >= 10 else None

# Compare with previous versions
print("\n" + "=" * 70)
print(f"  VERSION COMPARISON: {DATASET.upper()}")
print("=" * 70)

for vname, vfile in [("v2", "all_results.json"), ("v2.1", "v21_results.json")]:
    vpath = os.path.join(CACHE_DIR, vfile)
    if os.path.exists(vpath):
        with open(vpath) as f:
            vdata = json.load(f)
        if "warm" in vdata and vdata["warm"]:
            v_maes = [r["mae"] for r in vdata["warm"]]
            print(f"  {vname:>6s}:  MAE = {np.mean(v_maes):.4f} +/- {np.std(v_maes):.4f}")

v22_maes = [r["mae"] for r in v22_results]
print(f"  {'v2.2':>6s}:  MAE = {np.mean(v22_maes):.4f} +/- {np.std(v22_maes):.4f}")

if cold_user:
    print(f"\n  Cold-User: MAE={cold_user['mae']:.4f}  RMSE={cold_user['rmse']:.4f}")
if cold_item:
    print(f"  Cold-Item: MAE={cold_item['mae']:.4f}  RMSE={cold_item['rmse']:.4f}")

# Save
def jsonify(obj):
    if isinstance(obj, (np.floating,)):  return float(obj)
    if isinstance(obj, (np.integer,)):   return int(obj)
    if isinstance(obj, np.ndarray):      return obj.tolist()
    if isinstance(obj, dict):            return {k: jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):            return [jsonify(v) for v in obj]
    return obj

with open(os.path.join(CACHE_DIR, "v22_results.json"), "w") as f:
    json.dump(jsonify({"dataset": DATASET, "warm": v22_results,
                        "cold_user": cold_user, "cold_item": cold_item}), f, indent=2)
print(f"\nSaved to {CACHE_DIR}/v22_results.json")